将数据转换为json格式，供前端使用

读取数据

In [1]:
import numpy as np
import pandas as pd
import json
data_dict = np.load('../datasets/datasets.npy', allow_pickle=True).item()
works_df = data_dict['作品表']
people_df = data_dict['人物表']
events_df = data_dict['事件表']
location_df = data_dict['地点表']
poems_df = data_dict['金陵历朝诗歌']

data_dict_en = np.load('../datasets/datasets_en.npy', allow_pickle=True).item()
works_df_en = data_dict_en['作品表']
people_df_en = data_dict_en['人物表']
events_df_en = data_dict_en['事件表']
location_df_en = data_dict_en['地点表']
poems_df_en = data_dict_en['金陵历朝诗歌']

做一个新的事件列表（json格式），将二级地点的经纬度标出来

In [2]:
events_list = []
for row_id, row in events_df.iterrows():
    cur_category = row['事件类别']
    row_en = events_df_en[events_df_en['事件id'] == row['事件id']].iloc[0]
    if cur_category == '作品事件':
        continue
    cur_locations = row['二级地点']
    if pd.isna(cur_locations):
        continue
    cur_locations = cur_locations.strip()
    cur_locations = cur_locations.split('、')
    for location in cur_locations:
        # 如果地点表里面有，才进一步操作
        selected_location_df = location_df[location_df['地名'] == location]
        if selected_location_df.shape[0] == 1:
            if not pd.isna(selected_location_df.iloc[0]['经度']):
                cur_object = {}
                # '事件id', '事件名称', '朝代', '时期', '年号', '年份', '事件简介', '事件内容', '检索词', '事件类别', '特征类别', '一级地点', '二级地点', '相关人物', '相关作品'
                cur_object['事件id'] = None if pd.isna(row_en['事件id']) else int(row_en['事件id'])
                cur_object['事件名称'] = None if pd.isna(row_en['事件名称']) else row_en['事件名称']
                cur_object['朝代'] = None if pd.isna(row_en['朝代']) else row_en['朝代']
                cur_object['时期'] = None if pd.isna(row_en['时期']) else row_en['时期']
                # cur_object['地点'] = location_df_en[location_df_en['地名id'] == selected_location_df.iloc[0]['地名id']].iloc[0]['地名']
                cur_object['地点'] = location
                cur_object['经度'] = selected_location_df.iloc[0]['经度']
                cur_object['纬度'] = selected_location_df.iloc[0]['纬度']
                cur_object['事件简介'] = None if pd.isna(row_en['事件简介']) else row_en['事件简介']
                cur_object['事件内容'] = None if pd.isna(row_en['事件内容']) else row_en['事件内容']
                cur_object['检索词'] = None if pd.isna(row_en['检索词']) else row_en['检索词']
                cur_object['事件类别'] = None if pd.isna(row_en['事件类别']) else row_en['事件类别']
                cur_object['特征类别'] = None if pd.isna(row_en['特征类别']) else row_en['特征类别']
                cur_object['相关人物'] = None if pd.isna(row['相关人物']) else row['相关人物']
                cur_object['相关作品'] = None if pd.isna(row['相关作品']) else row['相关作品']
                events_list.append(cur_object)
events_list_json = {'events': events_list}
with open('../json/en/events_list.json', 'w') as f:
    json.dump(events_list_json, f)


做一个新的作品列表（json格式），将相关地点的经纬度标出来

In [3]:
# works_list = []
# for row_id, row in works_df.iterrows():
#     cur_locations = row['相关地点']
#     if pd.isna(cur_locations):
#         continue
#     cur_locations = cur_locations.strip()
#     cur_locations = cur_locations.split('、')
#     for location in cur_locations:
#         selected_location_df = location_df[location_df['地名'] == location]
#         if selected_location_df.shape[0] == 1:
#             if not pd.isna(selected_location_df.iloc[0]['经度']):
#                 cur_object = {}
#                 # '作品id', '作品名', '人物id', '名字', '朝代', '时期', '国家', '文体', '写作时间', '创作背景', '内容简介', '相关作品', '相关人物', '相关地点', '相关事件', '诗歌全文'
#                 cur_object['作品id'] = None if pd.isna(row['作品id']) else row['作品id']
#                 cur_object['作品名'] = None if pd.isna(row['作品名']) else row['作品名']
#                 cur_object['人物id'] = None if pd.isna(row['人物id']) else row['人物id']
#                 cur_object['名字'] = None if pd.isna(row['名字']) else row['名字']
#                 cur_object['朝代'] = None if pd.isna(row['朝代']) else row['朝代']
#                 cur_object['时期'] = None if pd.isna(row['时期']) else row['时期']
#                 cur_object['地点'] = location
#                 cur_object['经度'] = selected_location_df.iloc[0]['经度']
#                 cur_object['纬度'] = selected_location_df.iloc[0]['纬度']
#                 cur_object['国家'] = None if pd.isna(row['国家']) else row['国家']
#                 cur_object['文体'] = None if pd.isna(row['文体']) else row['文体']
#                 cur_object['创作背景'] = None if pd.isna(row['创作背景']) else row['创作背景']
#                 cur_object['内容简介'] = None if pd.isna(row['内容简介']) else row['内容简介']
#                 cur_object['相关作品'] = None if pd.isna(row['相关作品']) else row['相关作品']
#                 cur_object['相关人物'] = None if pd.isna(row['相关人物']) else row['相关人物']
#                 cur_object['相关事件'] = None if pd.isna(row['相关事件']) else row['相关事件']
#                 cur_object['诗歌全文'] = None if pd.isna(row['诗歌全文']) else row['诗歌全文']
#                 works_list.append(cur_object)
# works_list_json = {'works': works_list}
# with open('../json/works_list.json', 'w') as f:
#     json.dump(works_list_json, f)

根据朝代，地点，筛选出事件列表，生成一个这样的json文件

In [4]:
events_json = {}
for row_id, row in events_df.iterrows():
    cur_category = row['事件类别']
    row_en = events_df_en[events_df_en['事件id'] == row['事件id']].iloc[0]
    if cur_category == '作品事件':
        continue
    cur_locations = row['二级地点']
    if pd.isna(cur_locations):
        continue
    cur_locations = cur_locations.strip()
    cur_locations = cur_locations.split('、')
    cur_dynasty = row_en['朝代']
    for location in cur_locations:
        # 如果地点表里面有，才进一步操作
        selected_location_df = location_df[location_df['地名'] == location]
        if selected_location_df.shape[0] == 1:
            location_en = location_df_en[location_df_en['地名id'] == selected_location_df.iloc[0]['地名id']].iloc[0]['地名']
            if not pd.isna(selected_location_df.iloc[0]['经度']):
                cur_object = {}
                # '事件id', '事件名称', '朝代', '时期', '年号', '年份', '事件简介', '事件内容', '检索词', '事件类别', '特征类别', '一级地点', '二级地点', '相关人物', '相关作品'
                cur_object['事件id'] = None if pd.isna(row_en['事件id']) else int(row_en['事件id'])
                cur_object['事件名称'] = None if pd.isna(row_en['事件名称']) else row_en['事件名称']
                cur_object['朝代'] = None if pd.isna(row_en['朝代']) else row_en['朝代']
                cur_object['时期'] = None if pd.isna(row_en['时期']) else row_en['时期']
                cur_object['地点'] = location
                cur_object['事件简介'] = None if pd.isna(row_en['事件简介']) else row_en['事件简介']
                cur_object['事件内容'] = None if pd.isna(row_en['事件内容']) else row_en['事件内容']
                cur_object['检索词'] = None if pd.isna(row_en['检索词']) else row_en['检索词']
                cur_object['事件类别'] = None if pd.isna(row_en['事件类别']) else row_en['事件类别']
                cur_object['特征类别'] = None if pd.isna(row_en['特征类别']) else row_en['特征类别']
                cur_object['相关人物'] = None if pd.isna(row['相关人物']) else row['相关人物']
                cur_object['相关作品'] = None if pd.isna(row['相关作品']) else row['相关作品']
                location_x = selected_location_df.iloc[0]['经度']
                location_y = selected_location_df.iloc[0]['纬度']
                if cur_dynasty not in events_json.keys():
                    events_json[cur_dynasty] = {}
                if location not in events_json[cur_dynasty].keys():
                    events_json[cur_dynasty][location] = {'经度': location_x, '维度': location_y, 'events': []}
                events_json[cur_dynasty][location]['events'].append(cur_object)
with open('../json/en/dynasty_location_2_events.json', 'w') as f:
    json.dump(events_json, f)
                

根据朝代，地点，筛选出作品列表，生成一个这样的json文件

In [5]:
works_json = {}
for row_id, row in works_df.iterrows():
    cur_locations = row['相关地点']
    row_en = works_df_en[works_df_en['作品id'] == row['作品id']].iloc[0]
    if pd.isna(cur_locations):
        continue
    cur_locations = cur_locations.strip()
    cur_locations = cur_locations.split('、')
    cur_dynasty = row_en['朝代']
    for location in cur_locations:
        selected_location_df = location_df[location_df['地名'] == location]
        if selected_location_df.shape[0] == 1:
            location_en = location_df_en[location_df_en['地名id'] == selected_location_df.iloc[0]['地名id']].iloc[0]['地名']
            if not pd.isna(selected_location_df.iloc[0]['经度']):
                cur_object = {}
                # '作品id', '作品名', '人物id', '名字', '朝代', '时期', '国家', '文体', '写作时间', '创作背景', '内容简介', '相关作品', '相关人物', '相关地点', '相关事件', '诗歌全文'
                cur_object['作品id'] = None if pd.isna(row_en['作品id']) else int(row_en['作品id'])
                cur_object['作品名'] = None if pd.isna(row['作品名']) else row['作品名']
                cur_object['人物id'] = None if pd.isna(row_en['人物id']) else row_en['人物id']
                cur_object['名字'] = None if pd.isna(row['名字']) else row['名字']
                cur_object['朝代'] = None if pd.isna(row_en['朝代']) else row_en['朝代']
                cur_object['时期'] = None if pd.isna(row_en['时期']) else row_en['时期']
                cur_object['地点'] = location
                cur_object['国家'] = None if pd.isna(row_en['国家']) else row_en['国家']
                cur_object['文体'] = None if pd.isna(row['文体']) else row['文体']
                cur_object['创作背景'] = None if pd.isna(row_en['创作背景']) else row_en['创作背景']
                cur_object['内容简介'] = None if pd.isna(row_en['内容简介']) else row_en['内容简介']
                cur_object['相关作品'] = None if pd.isna(row_en['相关作品']) else row_en['相关作品']
                cur_object['相关人物'] = None if pd.isna(row['相关人物']) else row['相关人物']
                cur_object['相关事件'] = None if pd.isna(row_en['相关事件']) else row_en['相关事件']
                cur_object['诗歌全文'] = None if pd.isna(row['诗歌全文']) else row['诗歌全文']
                location_x = selected_location_df.iloc[0]['经度']
                location_y = selected_location_df.iloc[0]['纬度']
                if cur_dynasty not in works_json.keys():
                    works_json[cur_dynasty] = {}
                if location not in works_json[cur_dynasty].keys():
                    works_json[cur_dynasty][location] = {'经度': location_x, '维度': location_y, 'works': []}
                works_json[cur_dynasty][location]['works'].append(cur_object)
with open('../json/en/dynasty_location_2_works.json', 'w') as f:
    json.dump(works_json, f)

地点类别->地点列表，生成json

In [6]:
# import numpy as np
# import pandas as pd
# import json
# data_dict = np.load('../datasets/datasets.npy', allow_pickle=True).item()
# works_df = data_dict['作品表']
# people_df = data_dict['人物表']
# events_df = data_dict['事件表']
# location_df = data_dict['地点表']

In [7]:
location_category_dict = {}
dynasty_map = {'近代': '现当代', '六朝': '东吴', '明代': '明', '清朝': '清', '宋代': '宋', '唐代': '唐', '元代': '元'}
for row_id, row in location_df.iterrows():
    row_en = location_df_en[location_df_en['地名id'] == row['地名id']].iloc[0]
    cur_category = row['地名类别']
    if cur_category not in location_category_dict.keys():
        location_category_dict[cur_category] = []
    cur_object = {}
    # '地名id', '地名', '繁体地名', '别名', '地名类别', '景观类型', '地理位置', '朝代', '时期', '经度', '纬度', '相关作品', '相关人物', '相关地点', '相关事件', '地名描述'
    item_list = ['地名id', '地名', '繁体地名', '别名', '地名类别', '景观类型', '地理位置', '朝代', '时期', '经度', '纬度', '相关作品', '相关人物', '相关地点', '相关事件', '地名描述']
    for item in item_list:
        if item == '朝代':
            # cur_object[item] = dynasty_map[row[item]]
            cur_object[item] = row_en['朝代']
        elif item == '地名id':
            cur_object[item] = None if pd.isna(row_en[item]) else int(row_en[item])
        else:
            cur_object[item] = None if pd.isna(row_en[item]) else row_en[item]
    item_list = ['地名', '相关作品', '相关人物', '相关地点']
    for item in item_list:
        cur_object[item] = None if pd.isna(row[item]) else row[item]
    cur_object['作品数量'] = 0
    location_category_dict[cur_category].append(cur_object)
with open('../json/en/location_category.json', 'w') as f:
    json.dump(location_category_dict, f)

对于每一个地点加一个属性，作品数量

In [8]:
with open('../json/en/dynasty_location_2_works.json', 'r') as f:
    dynasty_location_2_works = json.load(f)

for category, location_list in location_category_dict.items():
    for location_object in location_list:
        cur_location = location_object['地名']
        for dynasty, location_2_works in dynasty_location_2_works.items():
            if cur_location in location_2_works.keys():
                location_object['作品数量'] += len(location_2_works[cur_location]['works'])
with open('../json/en/location_category.json', 'w') as f:
    json.dump(location_category_dict, f)
    

统计有多少地点的作品数量为0

In [9]:
# import json
# with open('../json/location_category.json', 'r') as f:
#     location_category_dict = json.load(f)
# zero_location_num = 0
# for _, location_list in location_category_dict.items():
#     for location_object in location_list:
#         if location_object['作品数量'] != 0:
#             zero_location_num += 1


把作品数量为0的去掉

In [10]:
import json
with open('../json/en/location_category.json', 'r') as f:
    location_category_dict_old = json.load(f)
location_category_dict_new = {}
for category, location_list_old in location_category_dict_old.items():
    location_list_new = []
    for location_object in location_list_old:
        if location_object['作品数量'] != 0:
            location_list_new.append(location_object)
    if len(location_list_new) != 0:
        location_category_dict_new[category] = location_list_new
with open('../json/en/location_category.json', 'w') as f:
    json.dump(location_category_dict_new, f)


统计各类型数量

In [11]:
import json
with open('../json/en/location_category.json', 'r') as f:
    location_category_dict = json.load(f)
category_num_list = []
for k, v in location_category_dict.items():
    category_num_list.append([k, len(v)])
category_num_list.sort(key= lambda x: x[1], reverse=True)
category_num_list

[['山脉', 24],
 ['水体', 15],
 ['寺观/祠庙', 9],
 ['古迹', 6],
 ['园林', 6],
 ['城池', 5],
 ['寺观', 5],
 ['码头/驿站', 4],
 ['桥梁', 4],
 ['城门', 4],
 ['宫殿', 4],
 ['道路', 4],
 ['行政区划', 3],
 ['陵墓', 3],
 ['祠庙', 3],
 ['居民区', 2],
 ['水利设施', 2],
 ['其他建筑', 1],
 ['衙署', 1],
 ['人工河渠', 1],
 ['学校', 1]]

合并上述类别：
- 山脉
- 古迹
- 园林
- 水体：水体、水利设施、人工河渠
- 官方建筑：城池、城门、宫殿、行政区划
- 寺观寺庙：寺观/祠庙、寺观、祠庙
- 建筑：码头/驿站、桥梁、陵墓、其他建筑
- 生活设施：道路、居民区、学校

In [12]:
import json
with open('../json/en/location_category.json', 'r') as f:
    location_category_dict = json.load(f)
location_category_dict_new = {}
location_category_dict_new['山脉'] = location_category_dict['山脉']
location_category_dict_new['古迹'] = location_category_dict['古迹']
location_category_dict_new['园林'] = location_category_dict['园林']
location_category_dict_new['水体'] = location_category_dict['水体'] + location_category_dict['水利设施'] + location_category_dict['人工河渠']
location_category_dict_new['官方建筑'] = location_category_dict['城池'] + location_category_dict['城门'] + location_category_dict['宫殿'] + location_category_dict['行政区划']
location_category_dict_new['寺观祠庙'] = location_category_dict['寺观/祠庙'] + location_category_dict['寺观'] + location_category_dict['祠庙']
location_category_dict_new['建筑'] = location_category_dict['码头/驿站'] + location_category_dict['桥梁'] + location_category_dict['陵墓'] + location_category_dict['其他建筑']
location_category_dict_new['生活设施'] = location_category_dict['道路'] + location_category_dict['居民区'] + location_category_dict['学校'] + location_category_dict['衙署']
with open('../json/en/location_category.json', 'w') as f:
    json.dump(location_category_dict_new, f)


更改其中的相关地点，把在本表中找不到对应词条的地点去掉，并且把与当前地点同类别的相关地点去掉，只保留跨类别的

In [13]:
# import json
# with open('../json/en/location_category.json', 'r') as f:
#     location_category_dict = json.load(f)
# category_list = ['山脉', '古迹', '园林', '水体', '官方建筑', '寺观祠庙', '建筑', '生活设施']
# location_dict = {}
# for category in category_list:
#     for location_object in location_category_dict[category]:
#         location_name = location_object['地名']
#         if location_name not in location_dict.keys():
#             location_dict[location_name] = category
# for category in category_list:
#     for i in range(len(location_category_dict[category])):
#         cur_relative_locations = location_category_dict[category][i]['相关地点']
#         if cur_relative_locations == None:
#             location_category_dict[category][i]['相关地点'] = []
#             continue
#         cur_relative_locations = cur_relative_locations.strip().split('、')
#         cur_relative_locations = list(filter(lambda x: x in location_dict.keys(), cur_relative_locations))
#         cur_relative_locations = list(filter(lambda x: location_dict[x] != category, cur_relative_locations))
#         location_category_dict[category][i]['相关地点'] = cur_relative_locations
# with open('../json/location_category.json', 'w') as f:
#     json.dump(location_category_dict, f)


再次筛选相关地点，按照经纬度计算直线距离，每个地点筛选出5个距离最近的相关地点 

In [14]:
# from geopy.distance import geodesic
# with open('../json/location_category.json', 'r') as f:
#     location_category_dict = json.load(f)
# category_list = ['山脉', '古迹', '园林', '水体', '官方建筑', '寺观祠庙', '建筑', '生活设施']
# location_weidu_jingdu_dict = {}
# for category in category_list:
#     for location_object in location_category_dict[category]:
#         location_name = location_object['地名']
#         if location_name not in location_weidu_jingdu_dict.keys():
#             location_weidu_jingdu_dict[location_name] = (location_object['纬度'], location_object['经度'])
# for category in category_list:
#     for i in range(len(location_category_dict[category])):
#         cur_location_name = location_category_dict[category][i]['地名']
#         cur_relative_locations = location_category_dict[category][i]['相关地点']
#         cur_relative_locations_distance = []
#         for relative_location in cur_relative_locations:
#             distance = geodesic(location_weidu_jingdu_dict[cur_location_name], location_weidu_jingdu_dict[relative_location]).m
#             cur_relative_locations_distance.append([relative_location, distance])
#         cur_relative_locations_distance = sorted(cur_relative_locations_distance, key=lambda x: x[1])
#         cur_relative_locations = list(map(lambda x: x[0], cur_relative_locations_distance))[:5]
#         location_category_dict[category][i]['相关地点'] = cur_relative_locations
# with open('../json/location_category.json', 'w') as f:
#     json.dump(location_category_dict, f)

# 直接按照中文版的对照过来
with open('../json/location_category.json', 'r') as f:
    location_category_dict = json.load(f)
with open('../json/en/location_category.json', 'r') as f:
    location_category_dict_en = json.load(f)
for category, location_list in location_category_dict.items():
    for i in range(len(location_list)):
        related_locations = location_category_dict[category][i]['相关地点']
        location_category_dict_en[category][i]['相关地点'] = []
        for related_location in related_locations:
            selected_location_df = location_df[location_df['地名'] == related_location].iloc[0]
            related_location_en = location_df_en[location_df_en['地名id'] == selected_location_df['地名id']].iloc[0]['地名']
            location_category_dict_en[category][i]['相关地点'].append(related_location)
with open('../json/en/location_category.json', 'w') as f:
    json.dump(location_category_dict_en, f)

统计各朝代作品和事件数量，（只统计能在地图上表出来的部分

In [15]:
import json
with open('../json/en/dynasty_location_2_works.json', 'r') as f:
    dynasty_location_2_works = json.load(f)
with open('../json/en/dynasty_location_2_events.json', 'r') as f:
    dynasty_location_2_events = json.load(f)
jsons = [dynasty_location_2_works, dynasty_location_2_events]
json_name = ['works', 'events']
id_name = ['作品id', '事件id']
dynasty_freq = {}
for cur_json_id, cur_json in enumerate(jsons):
    for dynasty, location_dict in cur_json.items():
        if dynasty not in dynasty_freq.keys():
            dynasty_freq[dynasty] = [0, 0]
        cur_item_id_list = []
        for location, location_detail in location_dict.items():
            cur_location_item_list = location_detail[json_name[cur_json_id]]
            for cur_object in cur_location_item_list:
                if cur_object[id_name[cur_json_id]] not in cur_item_id_list:
                    cur_item_id_list.append(cur_object[id_name[cur_json_id]])
        dynasty_freq[dynasty][cur_json_id] += len(cur_item_id_list)


In [16]:
dynasty_freq

{'Tang': [37, 3],
 'Song': [49, 5],
 'Yuan': [14, 0],
 'Qing': [71, 4],
 'Ming': [120, 20],
 'Southern Dynasties': [17, 12],
 'Southern Tang': [5, 0],
 'Modern': [16, 6],
 'Jin': [4, 7],
 'Eastern Wu': [0, 6],
 'Han': [0, 1]}

一级索引是诗歌集还是作品集

索引："work", "poem"

二级索引是id，作品id或者诗歌id，但是索引都是id

索引："id"

里面是诗歌的属性

- 包括诗的关键词：poems2keywords.npy，索引："keywords"
- 诗情感的积极消极打分：poems_scores.npy，索引："score"
- 情感的多元分析：杨雨彤做的，索引："sentiment1" "sentiment2"
- 诗的赏析：你自己做的，索引："appreciation"
- title
- content
- author

放在json里面 取名poems.json

如果数据有空缺，空缺数据位空字符串

In [17]:
poems2keywords = np.load('../datasets/poems/poems2keywords.npy', allow_pickle=True).item()['poems2keywords']
poems_scores = np.load('../datasets/poems/poems_scores.npy', allow_pickle=True).item()['scores']
work_shangxi = np.load('../datasets/work_shangxi.npy', allow_pickle=True).item()['work_shangxi']
print(poems2keywords.columns)
print(poems_scores.columns)
print(work_shangxi.columns)

Index(['诗歌作品id', '关键词'], dtype='object')
Index(['id', 'scores'], dtype='object')
Index(['作品id', '作品名', '人物id', '名字', '赏析', '是否来自原始数据集'], dtype='object')


In [18]:
poems_json = {'work': {}, 'poem': {}}
# work
for row_id, row in work_shangxi.iterrows():
    cur_object = {}
    cur_object['title'] = ''
    cur_object['author'] = ''
    cur_object['content'] = ''
    cur_object['keywords'] = ''
    cur_object['score'] = ''
    cur_object['sentiment1'] = ''
    cur_object['sentiment2'] = ''
    cur_object['appreciation'] = row['赏析']
    poems_json['work'][row['作品id']] = cur_object
# poem
for row_id, row in poems2keywords.iterrows():
    cur_object = {}
    cur_object['title'] = ''
    cur_object['author'] = ''
    cur_object['content'] = ''
    cur_object['keywords'] = row['关键词']
    cur_object['score'] = ''
    cur_object['sentiment1'] = ''
    cur_object['sentiment2'] = ''
    cur_object['appreciation'] = ''
    poems_json['poem'][row['诗歌作品id']] = cur_object
for row_id, row in poems_scores.iterrows():
    poems_json['poem'][row['id']]['score'] = row['scores']

In [19]:
with open('../json/en/poems.json', 'w') as f:
    json.dump(poems_json, f) 

生成works2keywords，补充到poems.json里面

In [20]:
import jieba
works2keywords_list = []
for row_id, row in works_df.iterrows():
    if pd.isna(row['诗歌全文']):
        continue
    content = row['诗歌全文']
    keywords_list = list(jieba.cut(content, cut_all=True))
    keywords_list = list(filter(lambda x: len(x) > 1, keywords_list))
    works2keywords_list.append([row['作品id'], keywords_list])
works2keywords_df = pd.DataFrame(data=works2keywords_list, columns=['作品id', '关键词'])
# works2keywords_df.to_csv('../bqs_only/works2keywords.csv', index=False)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\QISHUO~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.569 seconds.
Prefix dict has been built successfully.


In [21]:
with open('../json/en/poems.json', 'r') as f:
    poems_json = json.load(f)
for row_id, row in works2keywords_df.iterrows():
    id_str = str(row['作品id'])
    keywords_list = row['关键词']
    if id_str in poems_json['work'].keys():
        poems_json['work'][id_str]['keywords'] = keywords_list
    else:
        cur_object = {}
        cur_object['title'] = ''
        cur_object['author'] = ''
        cur_object['content'] = ''
        cur_object['keywords'] = keywords_list
        cur_object['score'] = ''
        cur_object['sentiment1'] = ''
        cur_object['sentiment2'] = ''
        cur_object['appreciation'] = ''
        poems_json['work'][id_str] = cur_object
with open('../json/en/poems.json', 'w') as f:
    json.dump(poems_json, f)

将情感分析 sentiment1 和 sentiment2 补充进poems.json

In [22]:
with open('../json/en/poems.json', 'r') as f:
    poems_json = json.load(f)
works_sentiment_df = pd.read_csv('../datasets/poems/作品表_带有诗歌全文_情感.csv')
poems_sentiment_df = pd.read_csv('../datasets/poems/金陵历朝诗歌_情感.csv')
print(works_sentiment_df.columns.tolist())
print(poems_sentiment_df.columns.tolist())

['作品id', '作品名', '人物id', '名字', '朝代', '时期', '国家', '文体', '写作时间', '创作背景', '内容简介', '相关作品', '相关人物', '相关地点', '相关事件', '诗歌全文', 'sentiment1', 'sentiment2']
['id', 'dynasty', 'author', 'title', 'content', 'sentiment1', 'sentiment2']


In [23]:
for row_id, row in works_sentiment_df.iterrows():
    id_str = str(row['作品id'])
    sentiment1 = row['sentiment1']
    sentiment2 = row['sentiment2']
    if id_str in poems_json['work'].keys():
        poems_json['work'][id_str]['sentiment1'] = sentiment1
        poems_json['work'][id_str]['sentiment2'] = sentiment2
    else:
        print('find new work! id: ' + id_str)
        cur_object = {}
        cur_object['title'] = ''
        cur_object['author'] = ''
        cur_object['content'] = ''
        cur_object['keywords'] = ''
        cur_object['score'] = ''
        cur_object['sentiment1'] = sentiment1
        cur_object['sentiment2'] = sentiment2
        cur_object['appreciation'] = ''
        poems_json['work'][id_str] = cur_object
for row_id, row in poems_sentiment_df.iterrows():
    id_str = str(row['id'])
    sentiment1 = row['sentiment1']
    sentiment2 = row['sentiment2']
    if id_str in poems_json['poem'].keys():
        poems_json['poem'][id_str]['sentiment1'] = sentiment1
        poems_json['poem'][id_str]['sentiment2'] = sentiment2
    else:
        print('find new poem! id: ' + id_str)
        cur_object = {}
        cur_object['title'] = ''
        cur_object['author'] = ''
        cur_object['content'] = ''
        cur_object['keywords'] = ''
        cur_object['score'] = ''
        cur_object['sentiment1'] = sentiment1
        cur_object['sentiment2'] = sentiment2
        cur_object['appreciation'] = ''
        poems_json['poem'][id_str] = cur_object

In [24]:
with open('../json/en/poems.json', 'w') as f:
    json.dump(poems_json, f)

顺便统计一下各类sentiment的频率

In [25]:
works_sentiment_df = pd.read_csv('../datasets/poems/作品表_带有诗歌全文_情感.csv')
poems_sentiment_df = pd.read_csv('../datasets/poems/金陵历朝诗歌_情感.csv')
sentiment1_dict = {}
sentiment2_dict = {}
for row_id, row in works_sentiment_df.iterrows():
    sentiment1 = row['sentiment1']
    sentiment2 = row['sentiment2']
    if sentiment1 not in sentiment1_dict.keys():
        sentiment1_dict[sentiment1] = 0
    if sentiment2 not in sentiment2_dict.keys():
        sentiment2_dict[sentiment2] = 0
    sentiment1_dict[sentiment1] += 1
    sentiment2_dict[sentiment2] += 1
print(sorted(sentiment1_dict.items(), key=lambda kv: (-kv[1], kv[0])))
print(sorted(sentiment2_dict.items(), key=lambda kv: (-kv[1], kv[0])))


[('思', 411), ('悲', 231), ('乐', 128), ('忧', 47), ('喜', 26), ('怒', 15)]
[('思', 306), ('乐', 209), ('悲', 204), ('忧', 81), ('喜', 36), ('怒', 22)]


用paddlehub的senta模型计算work中的作品情感score，补充到poems.json中

In [26]:
import paddlehub as hub
# 加载模型
senta = hub.Module(name="senta_lstm")

work_id_list = []
content_list = []

for row_id, row in works_df.iterrows():
    if pd.isna(row['诗歌全文']):
        continue
    content_list.append(row['诗歌全文'])
    work_id_list.append(row['作品id'])
score_list = senta.sentiment_classify(data={"text": content_list})
score_list = list(map(lambda x: x['positive_probs'], score_list))
works_scores_df = pd.DataFrame(data={'id': work_id_list, 'score': score_list})
# np.save('../datasets/works_scores.npy', {'works_scores': works_scores_df})
# works_scores_df.to_csv('../bqs_only/works_scores.csv', index=False)

c:\Users\QishuoBai\anaconda3\lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
[2023-09-28 14:38:43,611] [ WARNING] - The _initialize method in HubModule will soon be deprecated, you can use the __init__() to handle the initialization of the object


In [27]:
with open('../json/en/poems.json', 'r') as f:
    poems_json = json.load(f)
for row_id, row in works_scores_df.iterrows():
    id_str = str(int(row['id']))
    if id_str not in poems_json['work'].keys():
        print('find new id: ' + id_str)
        cur_object = {}
        cur_object['keywords'] = ''
        cur_object['score'] = row['score']
        cur_object['sentiment1'] = ''
        cur_object['sentiment2'] = ''
        cur_object['appreciation'] = ''
        poems_json['work'][id_str] = cur_object
    else:
        poems_json['work'][id_str]['score'] = row['score']
with open('../json/en/poems.json', 'w') as f:
    json.dump(poems_json, f)
        

在poems.json里面补充title、author、content

In [28]:
with open('../json/en/poems.json', 'r') as f:
    poems_json = json.load(f)
# work
for row_id, row in works_df.iterrows():
    row_en = works_df_en[works_df_en['作品id'] == row['作品id']].iloc[0]
    id_str = str(int(row['作品id']))
    if pd.isna(row['诗歌全文']):
        continue
    poems_json['work'][id_str]['title'] = '' if pd.isna(row['作品名']) else row['作品名']
    poems_json['work'][id_str]['author'] = '' if pd.isna(row['名字']) else row['名字']
    poems_json['work'][id_str]['content'] = row['诗歌全文']
# poem
for row_id, row in poems_df.iterrows():
    row_en = poems_df_en[poems_df_en['id'] == row['id']].iloc[0]
    id_str = str(int(row['id']))
    poems_json['poem'][id_str]['title'] = row['title']
    poems_json['poem'][id_str]['author'] = row['author']
    poems_json['poem'][id_str]['content'] = row['content']
with open('../json/en/poems.json', 'w') as f:
    json.dump(poems_json, f)

在poems.json里面补充location

In [29]:
with open('../json/en/poems.json', 'r') as f:
    poems_json = json.load(f)
# work
for row_id, row in works_df.iterrows():
    row_en = works_df_en[works_df_en['作品id'] == row['作品id']].iloc[0]
    if pd.isna(row['诗歌全文']):
        continue
    id_str = str(int(row['作品id']))
    poems_json['work'][id_str]['location'] = '' if pd.isna(row['相关地点']) else row['相关地点']
with open('../json/en/poems.json', 'w') as f:
    json.dump(poems_json, f)

在poems.json中补充朝代

In [30]:
with open('../json/en/poems.json', 'r') as f:
    poems_json = json.load(f)
# work
poems_json_work = poems_json['work']
for work_id, work_obj in poems_json_work.items():
    works_df_row = works_df_en[works_df_en['作品id'] == int(work_id)].iloc[0]
    poems_json['work'][work_id]['dynasty'] = works_df_row['朝代']
# poem
poems_json_poem = poems_json['poem']
for poem_id, poem_obj in poems_json_poem.items():
    poems_df_row = poems_df_en[poems_df_en['id'] == int(poem_id)].iloc[0]
    cur_dynasty = poems_df_row['dynasty']
    poems_json['poem'][poem_id]['dynasty'] = cur_dynasty
with open('../json/en/poems.json', 'w') as f:
    json.dump(poems_json, f)

所有包含那几个南京别名的作品，算个情感平均值  
金陵、 秣陵、 南京、 建康、 建业、 江宁、 白下、 应天、 集庆、归化、丹阳、上元、孝陵、建邺

In [31]:
import numpy as np
import pandas as pd
poems_scores = np.load('../datasets/poems/poems_scores.npy', allow_pickle=True).item()['scores']
nanjing_keywords_list = ['金陵', '秣陵', '南京', '建康', '建业', '江宁', '白下', '应天', '集庆', '归化', '丹阳', '上元', '孝陵', '建邺']
keyword_score_dict = {} # avg_score, num
for keyword in nanjing_keywords_list:
    keyword_score_dict[keyword] = [0 ,0]
# poem
for i in range(1300):
    score = poems_scores.iloc[i]['scores']
    title = poems_df.iloc[i]['title']
    content = poems_df.iloc[i]['content']
    for keyword in nanjing_keywords_list:
        if (keyword in title) or (keyword in content):
            keyword_score_dict[keyword][0] += score
            keyword_score_dict[keyword][1] += 1
# work
# for row_id, row in works_scores_df.iterrows():
#     id = int(row['id'])
#     score = row['score']
#     title = works_df[works_df['作品id'] == id].iloc[0]['作品名']
#     content = works_df[works_df['作品id'] == id].iloc[0]['诗歌全文']
#     for keyword in nanjing_keywords_list:
#         if (keyword in title) or (keyword in content):
#             keyword_score_dict[keyword][0] += score
#             keyword_score_dict[keyword][1] += 1
for keyword in nanjing_keywords_list:
    if keyword_score_dict[keyword][1] != 0:
        keyword_score_dict[keyword][0] = keyword_score_dict[keyword][0]/keyword_score_dict[keyword][1]
    print(keyword, ': ', keyword_score_dict[keyword][0], ' ', keyword_score_dict[keyword][1])

金陵 :  0.3604268907563023   238
秣陵 :  0.2833113636363637   44
南京 :  0.45545454545454545   11
建康 :  0.592575   12
建业 :  0.4209103448275862   29
江宁 :  0.4511235294117647   17
白下 :  0.5351222222222223   9
应天 :  0.3344833333333333   6
集庆 :  0.0464   1
归化 :  0   0
丹阳 :  0.2726125   40
上元 :  0.4864285714285715   14
孝陵 :  0.32043   10
建邺 :  0.18582500000000002   4


生成知识图谱所需的json，knowledge_map.json

将id改为 event-id, person-id, location-id, work-id

事件：
- 人物
- 地点
- 作品

人物：
- 人物
- 事件
- 地点
- 作品

地点：
- 人物
- 事件
- 地点
- 作品

作品：
- 人物
- 事件
- 地点
- 作品

共四个表，合并起来  内部用id索引

id:  
- name -> str
- person -> []
- event -> []
- location -> []
- work -> []

In [32]:
# import numpy as np
# import pandas as pd
# import json
# data_dict = np.load('../datasets/datasets.npy', allow_pickle=True).item()
# works_df = data_dict['作品表']
# people_df = data_dict['人物表']
# events_df = data_dict['事件表']
# location_df = data_dict['地点表']
# poems_df = data_dict['金陵历朝诗歌']
# print(events_df.columns.tolist())
# print(people_df.columns.tolist())
# print(location_df.columns.tolist())
# print(works_df.columns.tolist())
# print(poems_df.columns.tolist())

In [33]:
# person_type_map_df = pd.read_excel('../bqs_only/related_person_type_freq_merge.xlsx', sheet_name=0)
# person_type_merge_map = {}
# person_type_merge_freq = {}
# person_type_merge_list = []
# for row_id, row in person_type_map_df.iterrows():
#     person_type_merge_map[row['身份']] = row['合并类别']
#     if row['合并类别'] not in person_type_merge_freq.keys():
#         person_type_merge_freq[row['合并类别']] = 0
#     person_type_merge_freq[row['合并类别']] += int(row['频率'])
# person_type_merge_list = list(map(lambda x: x[0], person_type_merge_freq.items()))
# print(person_type_merge_freq)

In [34]:
# dynasty_map = {'东吴': '东吴', '晋': '晋', '南朝': '南朝', '唐': '唐', '南唐': '南唐', '宋': '宋', '元': '元', '明': '明', '清': '清', '现当代': '现当代'}

In [35]:
# event_map = {}
# person_map = {}
# location_map = {}
# work_map = {}
# related_person = {} # 能够所及到的人物id
# # 事件表
# for row_id, row in events_df.iterrows():
#     id_str = 'event-' + str(row['事件id'])
#     cur_object = {}
#     cur_object['id'] = id_str
#     cur_object['name'] = row['事件名称']
#     cur_object['type'] = 'event'
#     cur_object['person'] = []
#     cur_object['location'] = []
#     cur_object['event'] = []
#     cur_object['work'] = []
#     cur_object['info'] = []
#     cur_object['detail'] = []
#     # 基本信息
#     info_list = ['朝代', '时期', '年号', '年份', '事件类别']
#     info_show = ['朝代', '时期', '年号', '年份', '类别']
#     for i, d in enumerate(info_list):
#         if not pd.isna(row[d]):
#             if d == '时期':
#                 cur_object['info'].append([info_show[i], row[d].strip().split(';')[0]])
#             else:
#                 cur_object['info'].append([info_show[i], str(row[d])])
#     # detail 事件简介，事件内容
#     detail_list = ['事件简介', '事件内容']
#     for d in detail_list:
#         if not pd.isna(row[d]):
#             cur_object['detail'].append([d, row[d].replace('\n', '')])
#     # 相关地点
#     if not pd.isna(row['二级地点']):
#         relative_location = row['二级地点'].strip().split('、')
#         relative_location = list(set(relative_location))
#         for location in relative_location:
#             selected_location_df = location_df[location_df['地名'] == location]
#             if selected_location_df.shape[0] != 0 and (not pd.isna(selected_location_df.iloc[0]['经度'])):
#                 location_id = 'location-' + str(selected_location_df.iloc[0]['地名id'])
#                 cur_location_object = {'id': location_id, 'name': location};
#                 cur_object['location'].append(cur_location_object)
#     # 相关人物
#     if not pd.isna(row['相关人物']):
#         relative_person = row['相关人物'].strip().split('、')
#         relative_person = list(set(relative_person))
#         for person in relative_person:
#             selected_people_df = people_df[people_df['名字'] == person]
#             if selected_people_df.shape[0] != 0:
#                 person_id = 'person-' + str(selected_people_df.iloc[0]['人物id'])
#                 cur_person_object = {'id': person_id, 'name': person}
#                 cur_object['person'].append(cur_person_object)
#                 related_person[person_id] = 1
#     # 相关作品
#     if not pd.isna(row['相关作品']):
#         relative_work = row['相关作品'].strip().split('、')
#         relative_work = list(set(relative_work))
#         for work in relative_work:
#             selected_work_df = works_df[works_df['作品名'] == work]
#             if selected_work_df.shape[0] != 0:
#                 work_id = 'work-' + str(selected_work_df.iloc[0]['作品id'])
#                 cur_work_object = {'id': work_id, 'name': work}
#                 cur_object['work'].append(cur_work_object)
#     event_map[id_str] = cur_object
# # 人物表
# for row_id, row in people_df.iterrows():
#     id_str = 'person-' + str(row['人物id'])
#     cur_object = {}
#     cur_object['id'] = id_str
#     cur_object['name'] = row['名字']
#     cur_object['type'] = 'person'
#     cur_object['person'] = []
#     cur_object['location'] = []
#     cur_object['event'] = []
#     cur_object['work'] = []
#     cur_object['info'] = []
#     cur_object['detail'] = []
#     cur_object['person_type'] = 'none' # 没有身份的默认值
#     # 基本信息
#     info_list = ['朝代', '时期', '生年', '卒年']
#     info_show = ['朝代', '时期', '生年', '卒年']
#     for i, d in enumerate(info_list):
#         if not pd.isna(row[d]):
#             if d == '时期':
#                 cur_object['info'].append([info_show[i], row[d].strip().split(';')[0]])
#             else:
#                 cur_object['info'].append([info_show[i], str(row[d])])
#     # 身份
#     if not pd.isna(row['文学身份']):
#         person_type = row['文学身份'].strip().split('、')[0]
#         if person_type in person_type_merge_map.keys():
#             cur_object['person_type'] = person_type_merge_map[person_type]
#         else:
#             cur_object['person_type'] = 'none'
#     # 将身份信息添加进Info
#     if cur_object['person_type'] != 'none':
#         cur_object['info'].append(['身份', cur_object['person_type']])
#     else:
#         cur_object['info'].append(['身份', '暂无信息'])
#     # detail 基本信息，文学作品
#     detail_list = ['基本信息', '文学作品']
#     for d in detail_list:
#         if not pd.isna(row[d]):
#             cur_object['detail'].append([d, row[d].replace('\n', '')])
#     # 相关地点
#     if not pd.isna(row['相关地点']):
#         relative_location = row['相关地点'].strip().split('、')
#         relative_location = list(set(relative_location))
#         for location in relative_location:
#             selected_location_df = location_df[location_df['地名'] == location]
#             if selected_location_df.shape[0] != 0 and (not pd.isna(selected_location_df.iloc[0]['经度'])):
#                 location_id = 'location-' + str(selected_location_df.iloc[0]['地名id'])
#                 cur_location_object = {'id': location_id, 'name': location};
#                 cur_object['location'].append(cur_location_object)
#     # 相关人物
#     if not pd.isna(row['相关人物']):
#         relative_person = row['相关人物'].strip().split('、')
#         relative_person = list(set(relative_person))
#         for person in relative_person:
#             selected_people_df = people_df[people_df['名字'] == person]
#             if selected_people_df.shape[0] != 0:
#                 person_id = 'person-' + str(selected_people_df.iloc[0]['人物id'])
#                 cur_person_object = {'id': person_id, 'name': person}
#                 cur_object['person'].append(cur_person_object)
#                 related_person[person_id] = 1
#     # 相关事件
#     if not pd.isna(row['相关事件']):
#         relative_event = row['相关事件'].strip().split('、')
#         relative_event = list(set(relative_event))
#         for event in relative_event:
#             selected_event_df = events_df[events_df['事件名称'] == event]
#             if selected_event_df.shape[0] != 0:
#                 event_id = 'event-' + str(selected_event_df.iloc[0]['事件id'])
#                 cur_event_object = {'id': event_id, 'name': event}
#                 cur_object['event'].append(cur_event_object)
#     # 相关作品
#     selected_work_df = works_df[works_df['人物id'] == row['人物id']] # 所著作品
#     write_work = {}
#     for i in range(selected_work_df.shape[0]):
#         work_id = 'work-' + str(selected_work_df.iloc[i]['作品id'])
#         cur_work_object = {'id': work_id, 'name': selected_work_df.iloc[i]['作品名']}
#         cur_object['work'].append(cur_work_object)
#         write_work[selected_work_df.iloc[i]['作品名']] = 1 # 防止相关作品和所著作品重复
#     if not pd.isna(row['相关作品']):
#         relative_work = row['相关作品'].strip().split('、')
#         relative_work = list(set(relative_work))
#         for work in relative_work:
#             if work in write_work.keys():
#                 continue
#             selected_work_df = works_df[works_df['作品名'] == work]
#             if selected_work_df.shape[0] != 0:
#                 work_id = 'work-' + str(selected_work_df.iloc[0]['作品id'])
#                 cur_work_object = {'id': work_id, 'name': work}
#                 cur_object['work'].append(cur_work_object)
#     person_map[id_str] = cur_object
# # 地点表
# for row_id, row in location_df.iterrows():
#     id_str = 'location-' + str(row['地名id'])
#     cur_object = {}
#     cur_object['id'] = id_str
#     cur_object['name'] = row['地名']
#     cur_object['type'] = 'location'
#     cur_object['person'] = []
#     cur_object['location'] = []
#     cur_object['event'] = []
#     cur_object['work'] = []
#     cur_object['info'] = []
#     cur_object['detail'] = []
#     # detail 别名，地名描述
#     detail_list = ['别名', '地名描述']
#     for d in detail_list:
#         if not pd.isna(row[d]):
#             cur_object['detail'].append([d, row[d].replace('\n', '')])
#     # 相关地点
#     if not pd.isna(row['相关地点']):
#         relative_location = row['相关地点'].strip().split('、')
#         relative_location = list(set(relative_location))
#         for location in relative_location:
#             selected_location_df = location_df[location_df['地名'] == location]
#             if selected_location_df.shape[0] != 0 and (not pd.isna(selected_location_df.iloc[0]['经度'])):
#                 location_id = 'location-' + str(selected_location_df.iloc[0]['地名id'])
#                 cur_location_object = {'id': location_id, 'name': location};
#                 cur_object['location'].append(cur_location_object)
#     # 相关人物
#     if not pd.isna(row['相关人物']):
#         relative_person = row['相关人物'].strip().split('、')
#         relative_person = list(set(relative_person))
#         for person in relative_person:
#             selected_people_df = people_df[people_df['名字'] == person]
#             if selected_people_df.shape[0] != 0:
#                 person_id = 'person-' + str(selected_people_df.iloc[0]['人物id'])
#                 cur_person_object = {'id': person_id, 'name': person}
#                 cur_object['person'].append(cur_person_object)
#                 related_person[person_id] = 1
#     # 相关事件
#     if not pd.isna(row['相关事件']):
#         relative_event = row['相关事件'].strip().split('、')
#         relative_event = list(set(relative_event))
#         for event in relative_event:
#             selected_event_df = events_df[events_df['事件名称'] == event]
#             if selected_event_df.shape[0] != 0:
#                 event_id = 'event-' + str(selected_event_df.iloc[0]['事件id'])
#                 cur_event_object = {'id': event_id, 'name': event}
#                 cur_object['event'].append(cur_event_object)
#     # 相关作品
#     if not pd.isna(row['相关作品']):
#         relative_work = row['相关作品'].strip().split('、')
#         relative_work = list(set(relative_work))
#         for work in relative_work:
#             selected_work_df = works_df[works_df['作品名'] == work]
#             if selected_work_df.shape[0] != 0:
#                 work_id = 'work-' + str(selected_work_df.iloc[0]['作品id'])
#                 cur_work_object = {'id': work_id, 'name': work}
#                 cur_object['work'].append(cur_work_object)
#     location_map[id_str] = cur_object
# # 作品表
# for row_id, row in works_df.iterrows():
#     id_str = 'work-' + str(row['作品id'])
#     cur_object = {}
#     cur_object['id'] = id_str
#     cur_object['name'] = row['作品名']
#     cur_object['type'] = 'work'
#     cur_object['person'] = []
#     cur_object['location'] = []
#     cur_object['event'] = []
#     cur_object['work'] = []
#     cur_object['info'] = []
#     cur_object['detail'] = []
#     # 基本信息
#     info_list = ['名字', '朝代', '时期', '文体', '写作时间']
#     info_show = ['作者', '朝代', '时期', '文体', '创作时间']
#     for i, d in enumerate(info_list):
#         if not pd.isna(row[d]):
#             if d == '时期':
#                 cur_object['info'].append([info_show[i], row[d].strip().split(';')[0]])
#             else:
#                 cur_object['info'].append([info_show[i], str(row[d])])
#     # detail 名字，朝代，文体，创作背景，内容简介
#     detail_list = ['名字', '朝代', '创作背景', '内容简介']
#     for d in detail_list:
#         if not pd.isna(row[d]):
#             if d == '名字':
#                 cur_object['detail'].append(['作者', row[d].replace('\n', '')])
#             else:
#                 cur_object['detail'].append([d, row[d].replace('\n', '')])
#     # 相关地点
#     if not pd.isna(row['相关地点']):
#         relative_location = row['相关地点'].strip().split('、')
#         relative_location = list(set(relative_location))
#         for location in relative_location:
#             selected_location_df = location_df[location_df['地名'] == location]
#             if selected_location_df.shape[0] != 0 and (not pd.isna(selected_location_df.iloc[0]['经度'])):
#                 location_id = 'location-' + str(selected_location_df.iloc[0]['地名id'])
#                 cur_location_object = {'id': location_id, 'name': location};
#                 cur_object['location'].append(cur_location_object)
#     # 相关人物
#     if not pd.isna(row['相关人物']) or not pd.isna(row['名字']):
#         relative_person = []
#         if not pd.isna(row['相关人物']):
#             relative_person += row['相关人物'].strip().split('、')
#         if not pd.isna(row['名字']):
#             relative_person.append(row['名字'])
#         relative_person = list(set(relative_person))
#         for person in relative_person:
#             selected_people_df = people_df[people_df['名字'] == person]
#             if selected_people_df.shape[0] != 0:
#                 person_id = 'person-' + str(selected_people_df.iloc[0]['人物id'])
#                 cur_person_object = {'id': person_id, 'name': person}
#                 cur_object['person'].append(cur_person_object)
#                 related_person[person_id] = 1
#     # 相关事件
#     if not pd.isna(row['相关事件']):
#         relative_event = row['相关事件'].strip().split('、')
#         relative_event = list(set(relative_event))
#         for event in relative_event:
#             selected_event_df = events_df[events_df['事件名称'] == event]
#             if selected_event_df.shape[0] != 0:
#                 event_id = 'event-' + str(selected_event_df.iloc[0]['事件id'])
#                 cur_event_object = {'id': event_id, 'name': event}
#                 cur_object['event'].append(cur_event_object)
#     # 相关作品
#     if not pd.isna(row['相关作品']):
#         relative_work = row['相关作品'].strip().split('、')
#         relative_work = list(set(relative_work))
#         for work in relative_work:
#             selected_work_df = works_df[works_df['作品名'] == work]
#             if selected_work_df.shape[0] != 0:
#                 work_id = 'work-' + str(selected_work_df.iloc[0]['作品id'])
#                 cur_work_object = {'id': work_id, 'name': work}
#                 cur_object['work'].append(cur_work_object)
#     work_map[id_str] = cur_object

# # 合并
# knowledge_map = {'event': event_map, 'person': person_map, 'location': location_map, 'work': work_map, 'person_type_list': person_type_merge_list}

In [36]:
# with open('../json/knowledge_map.json', 'w') as f:
    # json.dump(knowledge_map, f)

统计图谱中涉及到的人物的身份频率

In [37]:
# person_type_freq = {}
# for person_id in related_person.keys():
#     id_num = int(person_id.split('-')[1])
#     row = people_df[people_df['人物id'] == id_num].iloc[0]
#     if pd.isna(row['文学身份']):
#         person_type = 'none'
#     else:
#         person_type = row['文学身份'].strip().split('、')[0]
#     if person_type not in person_type_freq.keys():
#         person_type_freq[person_type] = 0
#     person_type_freq[person_type] += 1
# person_type_freq_list = []
# for person_type, freq in person_type_freq.items():
#     person_type_freq_list.append([person_type, freq])
# person_type_freq_df = pd.DataFrame(data=person_type_freq_list, columns=['身份', '频率'])
# person_type_freq_df.sort_values(by='频率', ascending=False, inplace=True)
# person_type_freq_df.to_csv('../bqs_only/related_person_type_freq.csv', index=False)

统计整张图的node和link，添加到knowledge_map.json

In [38]:
# with open('../json/knowledge_map.json', 'r') as f:
#     knowledge_map = json.load(f)
# node_type_list = ['event', 'person', 'work']
# nodes_dict = {} # dict 防重复
# links_dict = {} # dict 防重复

# nodes = [] # id, name, type
# links = [] # souce, target
# for cur_type in node_type_list:
#     for source_id, source_object in knowledge_map[cur_type].items():
#         for target_type in node_type_list:
#             target_list = source_object[target_type]
#             for target_object in target_list:
#                 links_str_1 = source_id + ' ' + target_object['id']
#                 links_str_2 = target_object['id'] + ' ' + source_id
#                 if (links_str_1 not in links_dict.keys()) and (links_str_2 not in links_dict.keys()):
#                     links_dict[links_str_1] = 1
#                     links_dict[links_str_2] = 1
#                     links.append({'source': source_id, 'target': target_object['id']})
#                 if source_id not in nodes_dict.keys():
#                     nodes_dict[source_id] = 1
#                     nodes.append({'id': source_object['id'], 'name': source_object['name'], 'type': source_object['type']})
#                 if target_object['id'] not in nodes_dict.keys():
#                     nodes_dict[target_object['id']] = 1
#                     nodes.append({'id': target_object['id'], 'name': target_object['name'], 'type': target_type})
# knowledge_map['nodes'] = nodes
# knowledge_map['links'] = links
# with open('../json/knowledge_map.json', 'w') as f:
#     json.dump(knowledge_map, f)

#### keywords.json
键词到作品id的映射
json格式是一级目录是关键词，二级是"works","poems"，结果是id的列表  
还要"works_count", "poems_count", "total_count",  "average_score", "average_score_works", "average_score_poems"，多少作品中出现了这个关键词，这个关键词所有作品情感平均分多少

In [39]:
# # jieba 分词生成的所有关建词
# with open('../json/poems.json', 'r') as f:
#     poems_json = json.load(f)
# type_list = ['work', 'poem']
# keywords_json = {}
# for cur_type_name in type_list:
#     cur_data_dict = poems_json[cur_type_name]
#     for id, cur_object in cur_data_dict.items():
#         keywords_list = cur_object['keywords']
#         for keyword in keywords_list:
#             if keyword not in keywords_json.keys():
#                 keywords_json[keyword] = {}
#                 keywords_json[keyword]['works'] = []
#                 keywords_json[keyword]['poems'] = []
#                 keywords_json[keyword]['total_count'] = 0
#                 keywords_json[keyword]['average_score'] = 0
#                 keywords_json[keyword]['works_count'] = 0
#                 keywords_json[keyword]['average_score_works'] = 0
#                 keywords_json[keyword]['poems_count'] = 0
#                 keywords_json[keyword]['average_score_poems'] = 0
#             keywords_json[keyword][cur_type_name + 's'].append(id)
#             keywords_json[keyword]['total_count'] += 1
#             keywords_json[keyword]['average_score'] += cur_object['score']
#             keywords_json[keyword][cur_type_name + 's_count'] += 1
#             keywords_json[keyword]['average_score_' + cur_type_name + 's'] += cur_object['score']
# for keyword in keywords_json.keys():
#     keywords_json[keyword]['average_score'] /= keywords_json[keyword]['total_count']
#     keywords_json[keyword]['average_score'] = float(format(keywords_json[keyword]['average_score'], '.3f'))
#     if keywords_json[keyword]['works_count'] != 0:
#         keywords_json[keyword]['average_score_works'] /= keywords_json[keyword]['works_count']
#         keywords_json[keyword]['average_score_works'] = float(format(keywords_json[keyword]['average_score_works'], '.3f'))
#     if keywords_json[keyword]['poems_count'] != 0:
#         keywords_json[keyword]['average_score_poems'] /= keywords_json[keyword]['poems_count']
#         keywords_json[keyword]['average_score_poems'] = float(format(keywords_json[keyword]['average_score_poems'], '.3f'))
# with open('../json/keywords.json', 'w') as f:
#     json.dump(keywords_json, f)

把地点作为关建词加进去，或者覆盖进去

In [40]:
# location_score_dict = {}
# with open('../json/poems.json', 'r') as f:
#     poems_json = json.load(f)
# for row_id, row in works_df.iterrows():
#     if pd.isna(row['诗歌全文']):
#         continue
#     if pd.isna(row['相关地点']):
#         continue
#     relative_location = row['相关地点'].strip().split('、')
#     id_str = str(int(row['作品id']))
#     score = poems_json['work'][id_str]['score']
#     for location in relative_location:
#         if location_df[location_df['地名'] == location].shape[0] != 0 and (not pd.isna(location_df[location_df['地名'] == location].iloc[0]['经度'])):
#             if location not in location_score_dict.keys():
#                 location_score_dict[location] = {}
#                 location_score_dict[location]['works'] = []
#                 location_score_dict[location]['total_count'] = 0
#                 location_score_dict[location]['average_score'] = 0
#                 location_score_dict[location]['works_count'] = 0
#                 location_score_dict[location]['average_score_works'] = 0
#             location_score_dict[location]['works'].append(id_str)
#             location_score_dict[location]['total_count'] += 1
#             location_score_dict[location]['average_score'] += score
#             location_score_dict[location]['works_count'] += 1
#             location_score_dict[location]['average_score_works'] += score
# for location in location_score_dict.keys():
#     if location_score_dict[location]['total_count'] != 0:
#         location_score_dict[location]['average_score'] /= location_score_dict[location]['total_count']
#         location_score_dict[location]['average_score'] = float(format(location_score_dict[location]['average_score'], '.3f'))
#     if location_score_dict[location]['works_count'] != 0:
#         location_score_dict[location]['average_score_works'] /= location_score_dict[location]['works_count']
#         location_score_dict[location]['average_score_works'] = float(format(location_score_dict[location]['average_score_works'], '.3f'))
# for location in location_score_dict.keys():
#     if location not in keywords_json.keys():
#         keywords_json[location] = {}
#         keywords_json[location]['works'] = location_score_dict[location]['works']
#         keywords_json[location]['poems'] = []
#         keywords_json[location]['total_count'] = location_score_dict[location]['total_count']
#         keywords_json[location]['average_score'] = location_score_dict[location]['average_score']
#         keywords_json[location]['works_count'] = location_score_dict[location]['works_count']
#         keywords_json[location]['average_score_works'] = location_score_dict[location]['average_score_works']
#         keywords_json[location]['poems_count'] = 0
#         keywords_json[location]['average_score_poems'] = 0
#     else:
#         keywords_json[location]['works'] = location_score_dict[location]['works']
#         keywords_json[location]['total_count'] = location_score_dict[location]['total_count'] + keywords_json[location]['poems_count']
#         keywords_json[location]['average_score'] = (keywords_json[location]['poems_count'] * keywords_json[location]['average_score_poems'] + location_score_dict[location]['total_count'] * location_score_dict[location]['average_score'])/keywords_json[location]['total_count']
#         keywords_json[location]['average_score'] = float(format(keywords_json[location]['average_score'], '.3f'))
#         keywords_json[location]['works_count'] = location_score_dict[location]['works_count']
#         keywords_json[location]['average_score_works'] = location_score_dict[location]['average_score_works']
# with open('../json/keywords.json', 'w') as f:
#     json.dump(keywords_json, f)

更新关建词，加上部分关建词

In [41]:
# # 加入下列关建词
# core_keywords = ["石头", "鸡鸣", "雨花", "玄武", "金陵", "秦淮", "乌衣", "台城", "赏心", "钟山", "长干", "胭脂", "虎踞", "龙蟠", "六朝", "冶城", "新亭", "景阳", "长干曲", "王谢", "堂前燕", "陈后主", "张丽华", "韩擒虎", "后庭花", "金粉", "鸡鸣寺", "雨花台", "瞻园", "明孝陵", "丹阳", "上元", "孝陵", "建邺", "金陵",  "秣陵", "南京", "建康", "建业", "江宁", "白下", "应天", "集庆"];
# core_keywords = list(set(core_keywords))
# for keyword in core_keywords:
#     keywords_json[keyword] = {}
#     keywords_json[keyword]['works'] = []
#     keywords_json[keyword]['poems'] = []
#     keywords_json[keyword]['total_count'] = 0
#     keywords_json[keyword]['average_score'] = 0
#     keywords_json[keyword]['works_count'] = 0
#     keywords_json[keyword]['average_score_works'] = 0
#     keywords_json[keyword]['poems_count'] = 0
#     keywords_json[keyword]['average_score_poems'] = 0
    
# with open('../json/poems.json', 'r') as f:
#     poems_json = json.load(f)
# # work
# for row_id, row in works_df.iterrows():
#     if pd.isna(row['诗歌全文']):
#         continue
#     id_str = str(int(row['作品id']))
#     title = '' if pd.isna(row['作品名']) else row['作品名']
#     content = row['诗歌全文']
#     location = '' if pd.isna(row['相关地点']) else row['相关地点']
#     score = poems_json['work'][id_str]['score']
#     for keyword in core_keywords:
#         if (keyword in title) or (keyword in content) or (keyword in location):
#             keywords_json[keyword]['works'].append(id_str)
#             keywords_json[keyword]['total_count'] += 1
#             keywords_json[keyword]['average_score'] += score
#             keywords_json[keyword]['works_count'] += 1
#             keywords_json[keyword]['average_score_works'] += score
# # poem
# for row_id, row in poems_df.iterrows():
#     id_str = str(int(row['id']))
#     title = row['title']
#     content = row['content']
#     score = poems_json['poem'][id_str]['score']
#     for keyword in core_keywords:
#         if (keyword in title) or (keyword in content):
#             keywords_json[keyword]['poems'].append(id_str)
#             keywords_json[keyword]['total_count'] += 1
#             keywords_json[keyword]['average_score'] += score
#             keywords_json[keyword]['poems_count'] += 1
#             keywords_json[keyword]['average_score_poems'] += score
# for keyword in core_keywords:
#     if keywords_json[keyword]['total_count'] != 0:
#         keywords_json[keyword]['average_score'] /= keywords_json[keyword]['total_count']
#         keywords_json[keyword]['average_score'] = float(format(keywords_json[keyword]['average_score'], '.3f'))
#     else:
#         print(keyword)
#     if keywords_json[keyword]['works_count'] != 0:
#         keywords_json[keyword]['average_score_works'] /= keywords_json[keyword]['works_count']
#         keywords_json[keyword]['average_score_works'] = float(format(keywords_json[keyword]['average_score_works'], '.3f'))
#     if keywords_json[keyword]['poems_count'] != 0:
#         keywords_json[keyword]['average_score_poems'] /= keywords_json[keyword]['poems_count']
#         keywords_json[keyword]['average_score_poems'] = float(format(keywords_json[keyword]['average_score_poems'], '.3f'))
# with open('../json/keywords.json', 'w') as f:
#     json.dump(keywords_json, f)

重新生成一个keywords.json，里面只包括地名，一级索引为朝代，二级索引为关建词，内容包括两个average_score，一个表示当前朝代的平均值，另一个表示到当前朝代为止的平均值

In [42]:
# import numpy as np
# import pandas as pd
# import json
# import re
# data_dict = np.load('../datasets/datasets.npy', allow_pickle=True).item()
# works_df = data_dict['作品表']
# people_df = data_dict['人物表']
# events_df = data_dict['事件表']
# location_df = data_dict['地点表']
# poems_df = data_dict['金陵历朝诗歌']

# del_poems_list = []
# work_content_list = []
# for row_id, row in works_df.iterrows():
#     if pd.isna(row['诗歌全文']):
#         continue
#     work_content_list.append(re.sub('\W*', '', row['诗歌全文']))
# for row_id, row in poems_df.iterrows():
#     if re.sub('\W*', '', row['content']) in work_content_list:
#         del_poems_list.append(row['id'])
# len(del_poems_list)

# with open('../json/poems.json', 'r') as f:
#     poems_json = json.load(f)
# location_keywords = location_df['地名'].to_list()
# dynasty_list = ['东吴', '晋', '南朝', '唐', '南唐', '宋', '元', '明', '清', '现当代']
# dynasty_keywords_json = {}
# for dynasty in dynasty_list:
#     dynasty_keywords_json[dynasty] = {}
#     for location in location_keywords:
#         dynasty_keywords_json[dynasty][location] = {}
#         dynasty_keywords_json[dynasty][location]['cur_works'] = []
#         dynasty_keywords_json[dynasty][location]['cur_poems'] = []
#         dynasty_keywords_json[dynasty][location]['cur_count'] = 0
#         dynasty_keywords_json[dynasty][location]['cur_average_score'] = 0
#         dynasty_keywords_json[dynasty][location]['until_count'] = 0
#         dynasty_keywords_json[dynasty][location]['until_average_score'] = 0
# #work
# # work
# for row_id, row in works_df.iterrows():
#     if pd.isna(row['诗歌全文']):
#         continue
#     id_str = str(int(row['作品id']))
#     title = '' if pd.isna(row['作品名']) else row['作品名']
#     content = row['诗歌全文']
#     location = '' if pd.isna(row['相关地点']) else row['相关地点']
#     score = poems_json['work'][id_str]['score']
#     dynasty = poems_json['work'][id_str]['dynasty']
#     for keyword in location_keywords:
#         if (keyword in title) or (keyword in content) or (keyword in location):
#             dynasty_keywords_json[dynasty][keyword]['cur_works'].append(id_str)
#             dynasty_keywords_json[dynasty][keyword]['cur_count'] += 1
#             dynasty_keywords_json[dynasty][keyword]['cur_average_score'] += score
# # poem
# for row_id, row in poems_df.iterrows():
#     if row['id'] in del_poems_list:
#         continue
#     id_str = str(int(row['id']))
#     title = row['title']
#     content = row['content']
#     score = poems_json['poem'][id_str]['score']
#     dynasty = poems_json['poem'][id_str]['dynasty']
#     for keyword in location_keywords:
#         if (keyword in title) or (keyword in content):
#             dynasty_keywords_json[dynasty][keyword]['cur_poems'].append(id_str)
#             dynasty_keywords_json[dynasty][keyword]['cur_count'] += 1
#             dynasty_keywords_json[dynasty][keyword]['cur_average_score'] += score
# # 计算until
# for dynasty in dynasty_list:
#     for keyword in location_keywords:
#         if dynasty == '东吴':
#             dynasty_keywords_json[dynasty][keyword]['until_count'] = dynasty_keywords_json[dynasty][keyword]['cur_count']
#             dynasty_keywords_json[dynasty][keyword]['until_average_score'] = dynasty_keywords_json[dynasty][keyword]['cur_average_score']
#         else:
#             previous_dynasty = dynasty_list[dynasty_list.index(dynasty) - 1]
#             dynasty_keywords_json[dynasty][keyword]['until_count'] = dynasty_keywords_json[previous_dynasty][keyword]['until_count'] + dynasty_keywords_json[dynasty][keyword]['cur_count']
#             dynasty_keywords_json[dynasty][keyword]['until_average_score'] = dynasty_keywords_json[previous_dynasty][keyword]['until_average_score'] + dynasty_keywords_json[dynasty][keyword]['cur_average_score']
# # 取平均值
# for dynasty in dynasty_list:
#     for keyword in location_keywords:
#         dynasty_keywords_json[dynasty][keyword]['cur_average_score'] = dynasty_keywords_json[dynasty][keyword]['cur_average_score']/dynasty_keywords_json[dynasty][keyword]['cur_count'] if dynasty_keywords_json[dynasty][keyword]['cur_count']!=0 else 0
#         dynasty_keywords_json[dynasty][keyword]['until_average_score'] = dynasty_keywords_json[dynasty][keyword]['until_average_score']/dynasty_keywords_json[dynasty][keyword]['until_count'] if dynasty_keywords_json[dynasty][keyword]['until_count']!=0 else 0
# with open('../json/dynasty_keywords.json', 'w') as f:
#     json.dump(dynasty_keywords_json, f)

生成新的worklist，加入poem，并且修改相关性的判断标准，去掉poem中的已在work中存在的作品

In [43]:
# import numpy as np
# import pandas as pd
# import json
import re
# data_dict = np.load('../datasets/datasets.npy', allow_pickle=True).item()
# works_df = data_dict['作品表']
# people_df = data_dict['人物表']
# events_df = data_dict['事件表']
# location_df = data_dict['地点表']
# poems_df = data_dict['金陵历朝诗歌']

del_poems_list = []
work_content_list = []
for row_id, row in works_df.iterrows():
    if pd.isna(row['诗歌全文']):
        continue
    work_content_list.append(re.sub('\W*', '', row['诗歌全文']))
for row_id, row in poems_df.iterrows():
    if re.sub('\W*', '', row['content']) in work_content_list:
        del_poems_list.append(row['id'])
len(del_poems_list)

480

In [44]:
from tqdm import tqdm
with open('../json/en/poems.json', 'r') as f:
    poems_json = json.load(f)
works_list = []
location_names = location_df['地名'].to_list()
location_jingdu = location_df['经度'].to_list()
location_weidu = location_df['纬度'].to_list()
key_words=["建康","白下","江宁","建业","上元","南京","金陵","应天","秣陵","丹阳","孝陵"]
# work
for row_id, row in tqdm(works_df.iterrows()):
    row_en = works_df_en[works_df_en['作品id'] == row['作品id']].iloc[0]
    dynasty = row['朝代']
    if dynasty == '东晋':
        dynasty = '晋'
    dynasty_en = row_en['朝代']
    if pd.isna(row['诗歌全文']):
        # 无诗歌全文
        cur_locations = row['相关地点']
        if pd.isna(cur_locations):
            continue
        cur_locations = cur_locations.strip()
        cur_locations = cur_locations.split('、')
        for location in cur_locations:
            selected_location_df = location_df[location_df['地名'] == location]
            if selected_location_df.shape[0] == 1:
                # location_en = location_df_en[location_df_en['地名id'] == selected_location_df.iloc[0]['地名id']].iloc[0]['地名']
                if not pd.isna(selected_location_df.iloc[0]['经度']):
                    cur_object = {}
                    # '作品id', '作品名', '人物id', '名字', '朝代', '时期', '国家', '文体', '写作时间', '创作背景', '内容简介', '相关作品', '相关人物', '相关地点', '相关事件', '诗歌全文'
                    cur_object['作品id'] = 'work-' + str(row['作品id'])
                    cur_object['作品名'] = None if pd.isna(row['作品名']) else row['作品名']
                    cur_object['人物id'] = None if pd.isna(row_en['人物id']) else row_en['人物id']
                    cur_object['名字'] = None if pd.isna(row['名字']) else row['名字']
                    cur_object['朝代'] = dynasty_en
                    cur_object['时期'] = None if pd.isna(row_en['时期']) else row_en['时期']
                    cur_object['地点'] = location
                    cur_object['经度'] = selected_location_df.iloc[0]['经度']
                    cur_object['纬度'] = selected_location_df.iloc[0]['纬度']
                    cur_object['国家'] = None if pd.isna(row_en['国家']) else row_en['国家']
                    cur_object['文体'] = None if pd.isna(row['文体']) else row['文体']
                    cur_object['创作背景'] = None if pd.isna(row_en['创作背景']) else row_en['创作背景']
                    cur_object['内容简介'] = None if pd.isna(row_en['内容简介']) else row_en['内容简介']
                    cur_object['相关作品'] = None if pd.isna(row['相关作品']) else row['相关作品']
                    cur_object['相关人物'] = None if pd.isna(row['相关人物']) else row['相关人物']
                    cur_object['相关事件'] = None if pd.isna(row_en['相关事件']) else row_en['相关事件']
                    cur_object['诗歌全文'] = None if pd.isna(row['诗歌全文']) else row['诗歌全文']
                    cur_object['情感1'] = None
                    cur_object['情感2'] = None
                    cur_object['score'] = None
                    cur_object['关键词'] = None
                    works_list.append(cur_object)
    else:
        # 有诗歌全文
        title = '' if pd.isna(row['作品名']) else row['作品名']
        content = row['诗歌全文']
        location = '' if pd.isna(row['相关地点']) else row['相关地点']
        for loc_i, loc_name in enumerate(location_names):
            # loc_name_en = location_df_en.iloc[loc_i]['地名']
            if (loc_name in title) or (loc_name in content) or (loc_name in location):
                if not pd.isna(location_jingdu[loc_i]):
                    cur_object = {}
                    # '作品id', '作品名', '人物id', '名字', '朝代', '时期', '国家', '文体', '写作时间', '创作背景', '内容简介', '相关作品', '相关人物', '相关地点', '相关事件', '诗歌全文'
                    cur_object['作品id'] = 'work-' + str(row['作品id'])
                    cur_object['作品名'] = None if pd.isna(row['作品名']) else row['作品名']
                    cur_object['人物id'] = None if pd.isna(row_en['人物id']) else row_en['人物id']
                    cur_object['名字'] = None if pd.isna(row['名字']) else row['名字']
                    cur_object['朝代'] = dynasty_en
                    cur_object['时期'] = None if pd.isna(row_en['时期']) else row_en['时期']
                    cur_object['地点'] = loc_name
                    cur_object['经度'] = location_jingdu[loc_i]
                    cur_object['纬度'] = location_weidu[loc_i]
                    cur_object['国家'] = None if pd.isna(row_en['国家']) else row_en['国家']
                    cur_object['文体'] = None if pd.isna(row['文体']) else row['文体']
                    cur_object['创作背景'] = None if pd.isna(row_en['创作背景']) else row_en['创作背景']
                    cur_object['内容简介'] = None if pd.isna(row_en['内容简介']) else row_en['内容简介']
                    cur_object['相关作品'] = None if pd.isna(row['相关作品']) else row['相关作品']
                    cur_object['相关人物'] = None if pd.isna(row['相关人物']) else row['相关人物']
                    cur_object['相关事件'] = None if pd.isna(row_en['相关事件']) else row_en['相关事件']
                    cur_object['诗歌全文'] = None if pd.isna(row['诗歌全文']) else row['诗歌全文']
                    cur_object['情感1'] = poems_json['work'][str(row['作品id'])]['sentiment1']
                    cur_object['情感2'] = poems_json['work'][str(row['作品id'])]['sentiment2']
                    cur_object['score'] = poems_json['work'][str(row['作品id'])]['score']
                    cur_object['关键词'] = None
                    for k in key_words:
                        if k in content:
                            cur_object['关键词']=k
                    works_list.append(cur_object)
# poem
for row_id, row in tqdm(poems_df.iterrows()):
    if row['id'] in del_poems_list:
        continue
    row_en = poems_df_en[poems_df_en['id'] == row['id']].iloc[0]
    id_str = str(int(row['id']))
    title = row['title']
    content = row['content']
    sentiment1 = poems_json['poem'][id_str]['sentiment1']
    sentiment2 = poems_json['poem'][id_str]['sentiment2']
    dynasty = poems_json['poem'][id_str]['dynasty']
    for loc_i, loc_name in enumerate(location_names):
        # loc_name_en = location_df_en.iloc[loc_i]['地名']
        if (loc_name in title) or (loc_name in content):
            if not pd.isna(location_jingdu[loc_i]):
                cur_object = {}
                # '作品id', '作品名', '人物id', '名字', '朝代', '时期', '国家', '文体', '写作时间', '创作背景', '内容简介', '相关作品', '相关人物', '相关地点', '相关事件', '诗歌全文'
                cur_object['作品id'] = 'poem-' + id_str
                cur_object['作品名'] = row['title']
                cur_object['人物id'] = None
                cur_object['名字'] = None if pd.isna(row['author']) else row['author']
                cur_object['朝代'] = dynasty
                cur_object['时期'] = None
                cur_object['地点'] = loc_name
                cur_object['经度'] = location_jingdu[loc_i]
                cur_object['纬度'] = location_weidu[loc_i]
                cur_object['国家'] = None
                cur_object['文体'] = None
                cur_object['创作背景'] = None
                cur_object['内容简介'] = None
                cur_object['相关作品'] = None
                cur_object['相关人物'] = None
                cur_object['相关事件'] = None
                cur_object['诗歌全文'] = row['content']
                cur_object['情感1'] = sentiment1
                cur_object['情感2'] = sentiment2
                cur_object['score'] = poems_json['poem'][id_str]['score']
                cur_object['关键词'] = None
                for k in key_words:
                    if k in content:
                        cur_object['关键词']=k
                works_list.append(cur_object)
with open('../json/en/works_list_v2.json', 'w') as f:
    json.dump({'works': works_list}, f)

243it [00:00, 860.93it/s]

2072it [00:01, 1098.64it/s]
1300it [00:00, 1623.45it/s]


keywords_v2.json去重，与知识图谱的地点相关作品同步

In [45]:
# import numpy as np
# import pandas as pd
# import json
# data_dict = np.load('../datasets/datasets.npy', allow_pickle=True).item()
# works_df = data_dict['作品表']
# people_df = data_dict['人物表']
# events_df = data_dict['事件表']
# location_df = data_dict['地点表']
# poems_df = data_dict['金陵历朝诗歌']

# with open('../json/knowledge_map_v2.json', 'r') as f:
#     knowledge_map = json.load(f)
# with open('../json/poems.json', 'r') as f:
#     poems_json = json.load(f)
# with open('../json/keywords_v2.json', 'r') as f:
#     keywords_v2 = json.load(f)
# knowledge_map_location = knowledge_map['location']
# for location_id, location_obj in knowledge_map_location.items():
#     location_name = location_obj['name']
#     keywords_v2[location_name] = {}
#     keywords_v2[location_name]['works'] = []
#     keywords_v2[location_name]['poems'] = []
#     keywords_v2[location_name]['total_count'] = 0
#     keywords_v2[location_name]['average_score'] = 0
#     keywords_v2[location_name]['works_count'] = 0
#     keywords_v2[location_name]['average_score_works'] = 0
#     keywords_v2[location_name]['poems_count'] = 0
#     keywords_v2[location_name]['average_score_poems'] = 0
#     # work
#     for work in location_obj['work']:
#         work_id = work['id'].split('-')[1]
#         if work_id not in poems_json['work']:
#             continue
#         keywords_v2[location_name]['works'].append(work_id)
#         keywords_v2[location_name]['works_count'] += 1
#         keywords_v2[location_name]['average_score_works'] += poems_json['work'][work_id]['score']
#     # poem
#     for poem in location_obj['poem']:
#         poem_id = poem['id'].split('-')[1]
#         if poem_id not in poems_json['poem']:
#             continue
#         keywords_v2[location_name]['poems'].append(poem_id)
#         keywords_v2[location_name]['poems_count'] += 1
#         keywords_v2[location_name]['average_score_poems'] += poems_json['poem'][poem_id]['score']
#     keywords_v2[location_name]['total_count'] = keywords_v2[location_name]['works_count'] + keywords_v2[location_name]['poems_count']
#     keywords_v2[location_name]['average_score'] = keywords_v2[location_name]['average_score_works'] + keywords_v2[location_name]['average_score_poems']
#     # average
#     if keywords_v2[location_name]['total_count'] != 0:
#         keywords_v2[location_name]['average_score'] /= keywords_v2[location_name]['total_count']
#     if keywords_v2[location_name]['works_count'] != 0:
#         keywords_v2[location_name]['average_score_works'] /= keywords_v2[location_name]['works_count']
#     if keywords_v2[location_name]['poems_count'] != 0:
#         keywords_v2[location_name]['average_score_poems'] /= keywords_v2[location_name]['poems_count']
# with open('../json/keywords_v2.json', 'w') as f:
#     json.dump(keywords_v2, f)


生成location2id.json

In [46]:
# import numpy as np
# import pandas as pd
# import json
# data_dict = np.load('../datasets/datasets.npy', allow_pickle=True).item()
# works_df = data_dict['作品表']
# people_df = data_dict['人物表']
# events_df = data_dict['事件表']
# location_df = data_dict['地点表']
# poems_df = data_dict['金陵历朝诗歌']

location2obj = {}
with open('../json/en/location_category.json', 'r') as f:
    location_category = json.load(f)
for category, location_list in location_category.items():
    for location_obj in location_list:
        location2obj[location_obj['地名']] = location_obj
with open('../json/en/location2obj.json', 'w') as f:
    json.dump(location2obj, f)

翻译Keywords相关json文件

In [47]:
# location_zh2en= {}
# for row_id, row in location_df.iterrows():
#     row_en = location_df_en[location_df_en['地名id'] == row['地名id']].iloc[0]
#     location_zh2en[row['地名']] = row_en['地名']

In [48]:
# # dynasty_keywords.json翻译
# with open('../json/en/dynasty_keywords.json', 'r') as f:
#     dynasty_keywords_json = json.load(f)
# dynasty_keywords_json_new = {}
# for dynasty_en, keyword2obj in dynasty_keywords_json.items():
#     dynasty_keywords_json_new[dynasty_en] = {}
#     for keyword, keyword_obj in keyword2obj.items():
#         dynasty_keywords_json_new[dynasty_en][keyword] = keyword_obj
#         if keyword in location_zh2en:
#             dynasty_keywords_json_new[dynasty_en][location_zh2en[keyword]] = keyword_obj
# with open('../json/en/dynasty_keywords.json', 'w') as f:
#     json.dump(dynasty_keywords_json_new, f)

In [49]:
# # keywords_v2.json翻译
# with open('../json/en/keywords_v2.json', 'r') as f:
#     keywords_v2_json = json.load(f)
# keywords_v2_json_new = {}
# for keyword, keyword_obj in keywords_v2_json.items():
#     keywords_v2_json_new[keyword] = keyword_obj
#     if keyword in location_zh2en:
#         keywords_v2_json_new[location_zh2en[keyword]] = keyword_obj
# with open('../json/en/keywords_v2.json', 'w') as f:
#     json.dump(keywords_v2_json_new, f)

为event_list中的每个事件添加事件关键词

In [26]:
# LDA
# from gensim import corpora, models
# import jieba.posseg as jp, jieba
# import nltk
# from nltk.tokenize import word_tokenize
# from nltk.corpus import stopwords
# import string

# # 下载必要的数据
# nltk.download('punkt')
# nltk.download('stopwords')

# stop_words = set(stopwords.words('english'))

# def generate_event_keyword(events_list, i):
#     texts = []
#     for event in events_list:
#         name = event['事件名称'] if event['事件名称'] != None else ''
#         content = event['事件内容'] if event['事件内容'] != None else ''
#         texts.append(name + '.' + content)
#     words_list = []
#     punctuation = string.punctuation
#     punctuation += '\'\''
#     punctuation += '``'
#     punctuation += '\'s'
#     print(punctuation)
#     for sentence in texts:
#         tokens = word_tokenize(sentence)
#         filtered_tokens = [word for word in tokens if (word.lower() not in stop_words) and (word.lower() not in punctuation)]
#         words_list.append(filtered_tokens)
#     dictionary = corpora.Dictionary(words_list)
#     corpus = [dictionary.doc2bow(words) for words in words_list]
#     lda = models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=len(events_list))
#     # 打印所有主题，每个主题显示5个词
#     for topic in lda.print_topics(num_words=5):
#         print(topic)
#     # 主题推断
#     print(lda.inference(corpus))
#     return

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\QishuoBai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\QishuoBai\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [31]:
# textRank
# import networkx as nx
# from nltk.tokenize import word_tokenize
# from nltk.corpus import stopwords
# import string

# def preprocess_text(text):
#     # 分词
#     tokens = word_tokenize(text)
#     punctuation = string.punctuation
#     punctuation += '\'\''
#     punctuation += '``'
#     punctuation += '\'s'
    
#     # 去除停用词和标点符号
#     stop_words = set(stopwords.words('english'))
#     filtered_tokens = [word for word in tokens if (word.lower() not in stop_words) and (word.lower() not in punctuation)]
    
#     return filtered_tokens

# def build_graph(tokens):
#     # 创建图
#     G = nx.Graph()
    
#     # 添加节点
#     G.add_nodes_from(set(tokens))
    
#     # 添加边
#     for i in range(len(tokens) - 1):
#         for j in range(i + 1, len(tokens)):
#             G.add_edge(tokens[i], tokens[j])
    
#     return G

# def textrank_keywords(text, num_keywords=5):
#     # 预处理文本
#     tokens = preprocess_text(text)
    
#     # 构建图
#     G = build_graph(tokens)
    
#     # 计算 TextRank
#     pr = nx.pagerank(G)
    
#     # 获取排名靠前的关键词
#     keywords = sorted(pr, key=pr.get, reverse=True)[:num_keywords]
    
#     return keywords

# # # 示例文本
# # text = "Natural language processing is a subfield of artificial intelligence. It involves the interaction between computers and humans using natural language. TextRank is an algorithm for keyword and sentence extraction."

# # # 提取关键词
# # keywords = textrank_keywords(text)

# # # 打印结果
# # print("原始文本:", text)
# # print("提取的关键词:", keywords)

# def generate_event_keyword(events_list, i):
#     event = events_list[i]
#     name = event['事件名称'] if event['事件名称'] != None else ''
#     content = event['事件内容'] if event['事件内容'] != None else ''
#     keywords = textrank_keywords(name + '.' + content)
#     return keywords
    


In [40]:
# # TF-IDF
# from sklearn.feature_extraction.text import TfidfVectorizer
# from nltk.tokenize import word_tokenize
# from nltk.corpus import stopwords
# import string

# def preprocess_text(text):
#     # 分词
#     tokens = word_tokenize(text)
#     punctuation = string.punctuation
#     punctuation += '\'\''
#     punctuation += '``'
#     punctuation += '\'s'
    
#     # 去除停用词和标点符号
#     stop_words = stopwords.words('english')
#     stop_words = set(stop_words)
#     filtered_tokens = [word for word in tokens if (word.lower() not in stop_words) and (word.lower() not in punctuation)]
    
#     return ' '.join(filtered_tokens)

# def tfidf_keywords(text, num_keywords=5):
#     # 预处理文本
#     preprocessed_text = preprocess_text(text)
    
#     # 使用TF-IDF向量化器
#     vectorizer = TfidfVectorizer()
    
#     # 将文本转换为TF-IDF特征矩阵
#     tfidf_matrix = vectorizer.fit_transform([preprocessed_text])
    
#     # 获取关键词及其对应的TF-IDF值
#     feature_names = vectorizer.get_feature_names_out()
#     tfidf_scores = tfidf_matrix.toarray()[0]
    
#     # 按TF-IDF值排序，获取排名靠前的关键词
#     keywords = [feature_names[i] for i in tfidf_scores.argsort()[::-1][:num_keywords]]
    
#     return keywords

# # # 示例文本
# # text = "Natural language processing is a subfield of artificial intelligence. It involves the interaction between computers and humans using natural language. TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection or corpus."

# # # 提取关键词
# # keywords = tfidf_keywords(text)

# # # 打印结果
# # print("原始文本:", text)
# # print("提取的关键词:", keywords)

# def generate_event_keyword(events_list, i):
#     event = events_list[i]
#     name = event['事件名称'] if event['事件名称'] != None else ''
#     content = event['事件内容'] if event['事件内容'] != None else ''
#     keywords = tfidf_keywords(name + '.' + content)
#     return keywords


In [6]:
# gpt
import openai
import string

openai.api_key = 'sk-vXfPRJmW6CZ6r0fr599b84673003462b875b9d2bC4980b67'
openai.api_base = 'https://api.zhiyungpt.com/v1'

def gpt_keywords(q):
    rsp = openai.ChatCompletion.create(messages=[{"role": "user","content": q}],model="gpt-3.5-turbo")
    return rsp['choices'][0]['message']['content'].strip(string.punctuation)

def generate_event_keyword(events_list, i):
    event = events_list[i]
    name = ('事件名称：' + event['事件名称'] + ',') if event['事件名称'] != None else ''
    content = ('事件内容：' + event['事件内容']) if event['事件内容'] != None else ''
    q = '请你用不超过3个英文单词总结一下这个事件。' + name + content
    keywords = gpt_keywords(q)
    return keywords

In [7]:
import json
import time

with open('../json/en/events_list.json', 'r') as f:
    events_list = json.load(f)
events_list = events_list['events']

# print(generate_event_keyword(events_list, 0))

for i in range(28, len(events_list)):
    if i > 0 and events_list[i]['事件id'] == events_list[i-1]['事件id']:
        events_list[i]['keyword'] = events_list[i-1]['keyword']
        continue
    events_list[i]['keyword'] = generate_event_keyword(events_list, i)
    print(events_list[i]['keyword'])
    # print(generate_event_keyword(events_list, i))

with open('../json/en/events_list.json', 'w') as f:
    json.dump({'events': events_list}, f)

Symbolic burial
Ignored past connection
Accepting bribes openly
Ji Gang's Lakeside Adventure
Bohan's angry outburst
Liu Xie became monk
Confident unconventional choice
Absurd dragon incident
Qiu Fengjia's Deep Connection
Zhu Ziqing's Nanjing works
Pearl Buck's impact
Nanjing Poetry Gathering
Jiang Mingyu Society
Yecheng Grand Shrine: Gathering of Poets
Wanli painting club
Pan Zhiheng's Society
Jinling gathering
Society annual gathering
Literary society gathering
Louchuan Yaji Gathering
Qinhuai winter visits
Painting society founded
Poetic shrine unveiling
Yu Anqi joined Chunshe
Qinhuai Society formed
Qinhuai Poetry Club
Buddhism flourishes
Emperor's Fortunate Visit
Stone City Magnificent
Strategic military stronghold
Strategic fortress above river
River moves west
Empty city" poem
Mystical snowfall event
Shogunate established by Wang Dao
Renamed economic opportunity
Temple's name origin
Floral Rain Lecture
Temple fires devastate
Empress's temple stay
Love across Rivers
